# K-means clustering of LA-ICP-MS clinopyroxene trace-elements – La Palma (Cumbre Vieja)

**Caracciolo et al.**

**-Clustering code-**

This notebook contains the code used to apply k-means clustering and related tests to trace-element concentrations of clinopyroxene crystals from three Cumbre Vieja (La Palma) eruptions: **El Charco 1712**, **Teneguía 1971**, and
**Tajogaite 2021** using calibrated LA-ICP-MS element maps (pixels) together with conventional
single-spot analyses.

**Workflow**

1. **Imports**
2. **Configuration**: Every path, sample, and parameter used in the code is set up here.
3. **Helper functions**: loading, masking, preprocessing, and plotting functions.
4. **Data loading & preprocessing**: mask background and outlier pixels,
   replace near-zero values with noise.
5. **Combine & normalise**: pool all valid pixels and spots, log10-transform and
   standardise.
6. **Elbow method**: Evaluate and choose the final number of clusters K.
7. **K-means + PCA**: final clustering with stable labels, visualised in PCA space.
8. **t-SNE**: non-linear embedding of the clusters as an independent check.
9. **Stability analysis**: verify the clusters are robust to dataset composition.
10. **Clustered maps**: colour each crystal map with its cluster labels.

The notebook is fully self-contained: it only needs the Python packages listed in
the accompanying README and the Zenodo dataset folder `Maps_SingleSpots_LaPalma`
placed next to this notebook.

*Note:* Run all cells in order in a single pass (Restart & Run All). Some steps share a random generator and re-running cells out of order will change the random draws.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import string
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

# Matplotlib export settings: editable text in PDF/SVG, Arial font
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"

## 2. Configuration

Everything you might want to change is in this cell: the data location, the
element channels, the sample list, the masking/preprocessing parameters and the
cluster colours.

In [ ]:
# https://doi.org/10.5281/zenodo.17635257 — keep the original folder

DATA_ROOT  = Path("./Maps_SingleSpots_LaPalma")   # dataset root
MAP_SUBDIR = "time00000_level00000"               # subfolder with the channel CSVs
FIGURE_DIR = Path("./Figures")                    # all output figures go here


CHANNEL_FILES = {
    "Ca": "channel_Ca _42Ca_",
    "Cr": "channel_Cr _52Cr_",
    "Sc": "channel_Sc _45Sc_",
    "Zr": "channel_Zr _90Zr_",
    "Mn": "channel_Mn _55Mn_",
    "Ni": "channel_Ni _60Ni_",
    "Sr": "channel_Sr _88Sr_",
    "V":  "channel_V _51V_",
    "Ti": "channel_Ti _47Ti_",
    "Ce": "channel_Ce _140Ce_",
}

# The Ca channel is used only for masking: pixels with Ca counts below
# this threshold are background (epoxy / holes) and are excluded.
MASK_ELEMENT = "Ca"
CA_BACKGROUND_THRESHOLD = 1e5

# Clustering parameters
RANDOM_SEED = 42 # seed for every stochastic step -> reproducible
rng = np.random.default_rng(RANDOM_SEED)

# Elements used as clustering features.
# NOTE: the order matters for exact reproducibility. The stable relabelling below sorts clusters by the FIRST feature (scaled Cr).
ELEMENTS = ["Cr", "Sc", "Zr", "Mn", "Ni", "Sr", "V", "Ti"]

# Near-zero thresholds (ppm) per element: values below are replaced with
# uniform random noise in [0, threshold] to avoid log10(0) artefacts.
NEAR_ZERO_THRESHOLDS = {
    "Cr": 5, "Sc": 1, "Zr": 1, "Mn": 2,
    "Ni": 3, "Sr": 1, "V": 1, "Ti": 2,
}

# Percentile window used to flag extreme outlier pixels in the maps. A pixel is masked if ANY element falls outside this window.
OUTLIER_PERCENTILES = (0.1, 99.9)

# Number of clusters used in the publication (supported by the elbow plot below)
K_FINAL = 6

# Samples: one entry per crystal map
SAMPLES = [
    # --- El Charco 1712 ---
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 12",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63660_cpx12",
         scale_um_per_px=4.2, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 13",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63663_cpx13",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 1-2",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63664B-cpx1-2",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 21",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63671_cpx21",
         scale_um_per_px=4, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 22",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63671_cpx22",
         scale_um_per_px=5, bar_loc="bottom_right"),
    # --- Teneguia 1971 ---
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 4",
         path="Teneguia1971/Maps_1971/export_calibrated 1_43440_cpx4",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 3-5",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44136_cpx3-5",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 1",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44149_cpx1",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 3",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44149_cpx3",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 2-5",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44154_cpx2-5",
         scale_um_per_px=3, bar_loc="bottom_right"),
    # --- Tajogaite 2021 ---
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 3",
         path="2021LaPalma/Maps/118424-cpx3/export_calibrated 1_118424-cpx3",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 2",
         path="2021LaPalma/Maps/118465_cpx2/export_calibrated 1_118465-cpx2",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 0",
         path="2021LaPalma/Maps/118465_Test cpx/export_calibrated 1_Area",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 4",
         path="2021LaPalma/Maps/118508_cpx4/export_calibrated 1_118508-cpx4_Map",
         scale_um_per_px=5, bar_loc="bottom_left"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 5",
         path="2021LaPalma/Maps/118537_cpx5/export_calibrated 1_118537-cpx5",
         scale_um_per_px=3, bar_loc="bottom_left"),
]

# Single-spot analyses exported in the same folder structure as the maps.
# They join the clustering but are NOT maps (no outlier masking, no map plots).
SINGLE_SPOT_SAMPLE = dict(
    eruption="Single_spots", title="Single spots",
    path="Single spot_Clustering/SingleSpot_MapStructure",
    scale_um_per_px=None, bar_loc=None,
)

CLUSTER_COLOR_DICT = {
    0: "#E69F00",  # orange
    1: "#7FC9F5",  # sky blue
    2: "#009E73",  # green
    3: "#F0E442",  # yellow
    4: "#0072B2",  # blue
    5: "#D55E00",  # vermillion
}
BACKGROUND_COLOR = "#000000"   # masked / background pixels in cluster maps

print(f"Data root          : {DATA_ROOT.resolve()}")
print(f"Clustering features: {ELEMENTS}")
print(f"Samples            : {len(SAMPLES)} crystal maps + 1 single-spot set")

## 3. Helper functions

Functions used in sections below: channel loading with shape validation, background/outlier masking, near-zero noise replacement, figure saving, and a shared scatter-plot routine used for both the PCA and the t-SNE visualisations.

In [ ]:
# Loading & masking
def channel_csv_path(sample, element):
    return DATA_ROOT / sample["path"] / MAP_SUBDIR / f"{CHANNEL_FILES[element]}.csv"

def load_cube(sample, elements):
    """Load the requested element channels of one sample.
    Returns an array of shape (rows, cols, len(elements)), in the order
    given by 'elements'. Raises if a CSV file is missing or if the
    channel images do not all share the same spatial dimensions.
    """
    loaded = []
    for el in elements:
        fp = channel_csv_path(sample, el)
        if not fp.is_file():
            raise FileNotFoundError(
                f"Channel file not found: {fp}\n"
                f"Check that DATA_ROOT ({DATA_ROOT.resolve()}) points to the "
                f"Zenodo dataset and that the folder structure was not modified.")
        loaded.append(pd.read_csv(fp, header=None).values.astype(float))

    first_shape = loaded[0].shape
    for el, arr in zip(elements, loaded):
        if arr.shape != first_shape:
            raise ValueError(
                f"Channel shape mismatch in sample {sample['path']!r}: "
                f"{el} has shape {arr.shape}, expected {first_shape}.")

    return np.stack(loaded, axis=-1)

def background_mask(ca_channel, threshold=CA_BACKGROUND_THRESHOLD):
    return ca_channel < threshold

def save_figure(fig, name, formats=("pdf",), dpi=300, transparent=True):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for ext in formats:
        out = FIGURE_DIR / f"{name}.{ext}"
        fig.savefig(out, bbox_inches="tight", format=ext,
                    dpi=dpi, transparent=transparent)
        print(f"Saved: {out}")

def isotope_label(element):
    """Isotope label (e.g. '52Cr') from the channel CSV file name."""
    return CHANNEL_FILES[element].split("_")[-2]

def outlier_mask(cube, percentiles=OUTLIER_PERCENTILES):
    """Mask pixels that are extreme outliers in ANY channel of the cube.
    Percentiles are computed per channel over the full image. 'cube'
    must contain only the data channels (no Ca masking channel).
    """
    lower_p, upper_p = percentiles
    mask = np.zeros(cube.shape[:2], dtype=bool)
    for i in range(cube.shape[2]):
        lower = np.percentile(cube[:, :, i], lower_p)
        upper = np.percentile(cube[:, :, i], upper_p)
        mask |= (cube[:, :, i] < lower) | (cube[:, :, i] > upper)
    return mask

def replace_near_zero_with_noise(cube, elements, rng,
                                 thresholds=NEAR_ZERO_THRESHOLDS):
    """Replace near-zero values with uniform random noise in [0, threshold].
    Prevents log10(0) artefacts. 'elements' gives the element of each
    channel of 'cube'.
    """
    for i, el in enumerate(elements):
        t = thresholds[el]
        below = cube[:, :, i] < t
        noise = rng.uniform(low=0, high=t, size=cube[:, :, i].shape)
        cube[:, :, i] = np.where(below, noise, cube[:, :, i])
    return cube

# Cluster plotting
def cluster_cmap(k=K_FINAL):
    """Discrete colormap with one colour per cluster (no background)."""
    return ListedColormap([CLUSTER_COLOR_DICT[i] for i in range(k)])


def cluster_map_cmap_norm(k=K_FINAL):
    """Colormap + norm for 2-D cluster maps, with background as extra bin."""
    colors = [CLUSTER_COLOR_DICT[i] for i in range(k)] + [BACKGROUND_COLOR]
    cmap = ListedColormap(colors)
    bounds = np.arange(0, k + 2) - 0.5     # bin edges centred on integer labels
    norm = BoundaryNorm(boundaries=bounds, ncolors=len(colors))
    return cmap, norm


def plot_cluster_scatter(coords, labels, spot_mask, k=K_FINAL,
                         centroids_2d=None, xlabel="", ylabel="", title="",
                         xlim=None, ylim=None, pixel_alpha=0.15,
                         figsize=(7, 6.5)):
    color_kwargs = dict(cmap=cluster_cmap(k), vmin=-0.5, vmax=k - 0.5)

    fig, ax = plt.subplots(figsize=figsize)

    # Map pixels
    ax.scatter(coords[~spot_mask, 0], coords[~spot_mask, 1],
               c=labels[~spot_mask], s=6, marker="x", alpha=pixel_alpha,
               linewidths=0.3, zorder=1, **color_kwargs)

    # Single spots
    ax.scatter(coords[spot_mask, 0], coords[spot_mask, 1],
               c=labels[spot_mask], s=70, marker="D", edgecolor="black",
               linewidth=0.6, alpha=0.95, zorder=3, **color_kwargs)

    # Centroids
    if centroids_2d is not None:
        ax.scatter(centroids_2d[:, 0], centroids_2d[:, 1], s=240,
                   facecolors="none", edgecolors="black", linewidths=1.6,
                   marker="o", zorder=4)
        for i in range(k):
            ax.text(centroids_2d[i, 0] + 0.20, centroids_2d[i, 1] + 0.20,
                    str(i), fontsize=9, ha="left", va="bottom",
                    color="black", zorder=6)

    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.grid(linestyle="--", alpha=0.12)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

    # Legends: clusters (upper right) and point types (lower left)
    cluster_handles = [Patch(facecolor=CLUSTER_COLOR_DICT[i], edgecolor="none",
                             label=f"Cluster {i}") for i in range(k)]
    type_handles = [
        Line2D([0], [0], marker="x", color="black", linestyle="None",
               markersize=6, label="Pixels"),
        Line2D([0], [0], marker="D", color="w", markerfacecolor="lightgrey",
               markeredgecolor="black", linestyle="None", markersize=7,
               label="Single spots"),
    ]
    if centroids_2d is not None:
        type_handles.append(
            Line2D([0], [0], marker="o", color="black", markerfacecolor="none",
                   linestyle="None", markersize=10, label="Centroids"))

    leg1 = ax.legend(handles=cluster_handles, title="Cluster",
                     loc="upper right", fontsize=8, title_fontsize=9,
                     frameon=True)
    ax.add_artist(leg1)
    ax.legend(handles=type_handles, loc="lower left", fontsize=8, frameon=True)

    fig.tight_layout()
    return fig, ax

## 4. Data loading & per-map preprocessing

For every sample we:

* load the **Ca channel** (used only for masking) plus the eight clustering elements
  into a `(rows, cols, channels)` cube;
* **mask background pixels** (epoxy, holes) where Ca counts fall below
  `CA_BACKGROUND_THRESHOLD`;
* for crystal *maps* (not the single-spot set), additionally **mask extreme outliers**
  : a pixel is removed if any element falls outside the 0.1–99.9 percentile range;
* **replace near-zero values** with uniform random noise in `[0, threshold]`
  (per-element thresholds), which prevents `log10(0)` artefacts later;
* show a quick-look log-scale image of every elemental map (masked pixels in black);
* flatten the cube and keep only the valid pixels for clustering (the Ca channel is
  dropped at this point).

In [ ]:
# Samples entering the clustering: 15 crystal maps + the single-spot set
clustering_samples = SAMPLES + [SINGLE_SPOT_SAMPLE]

cube_info_list = []

for sample in clustering_samples:
    is_spot = sample is SINGLE_SPOT_SAMPLE

    # Load Ca (mask channel) + clustering elements -> (rows, cols, channels)
    cube = load_cube(sample, [MASK_ELEMENT] + ELEMENTS)
    n_rows, n_cols, n_channels = cube.shape
    print("loaded", sample["title"], cube.shape)

    # Background mask from the Ca channel
    main_mask = background_mask(cube[:, :, 0])

    # Maps only: also mask pixels that are extreme outliers in ANY element
    if not is_spot:
        main_mask |= outlier_mask(cube[:, :, 1:])

    # Replace near-zero values with uniform noise (avoids log10(0))
    cube[:, :, 1:] = replace_near_zero_with_noise(cube[:, :, 1:], ELEMENTS, rng)

    # Quick-look visualisation of each elemental map (log10 scale, masked = black)
    if not is_spot:
        cmap = plt.get_cmap("magma").copy()
        cmap.set_bad(color="black")
        fig, axes = plt.subplots(1, len(ELEMENTS), figsize=(15, 5))
        for idx, el in enumerate(ELEMENTS):
            masked = np.ma.masked_where(main_mask, cube[:, :, idx + 1])
            im = axes[idx].imshow(np.ma.log10(masked), cmap=cmap,
                                  interpolation="nearest")
            axes[idx].set_title(el)
            fig.colorbar(im, ax=axes[idx])
        plt.suptitle(sample["title"], fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()

    # Flatten to (pixels, channels) and keep only valid (unmasked) pixels.
    # The Ca channel is used only for masking and is dropped here.
    data_flat = cube.reshape(n_rows * n_cols, n_channels)
    mask_flat = main_mask.flatten()
    cube_info_list.append({
        "sample":          sample,
        "n_rows":          n_rows,
        "n_cols":          n_cols,
        "n_pixels_total":  data_flat.shape[0],
        "valid_indices":   np.where(~mask_flat)[0],
        "clustering_data": data_flat[~mask_flat][:, 1:],
        "is_spot":         is_spot,
    })

## 5. Combine & normalise

All valid pixels and single spots are pooled into one matrix
(`n_points × 8 elements`). Concentrations are **log10-transformed** and
**standardised** (z-scores), so each element contributes equally to the
Euclidean distances used by k-means.

In [ ]:
# Combine valid pixels/spots from all samples 
combined_data = np.vstack([info["clustering_data"] for info in cube_info_list])
combined_source = np.concatenate([np.full(info["clustering_data"].shape[0], info["is_spot"])
    for info in cube_info_list
])
print("Initial combined data shape:", combined_data.shape)

# Remove rows containing NaNs 
nan_mask = ~np.isnan(combined_data).any(axis=1)
n_dropped_nan = int((~nan_mask).sum())
combined_data, combined_source = combined_data[nan_mask], combined_source[nan_mask]
print(f"After NaN removal: {combined_data.shape}  ({n_dropped_nan} rows dropped)")

# log10 transform + z-score standardisation 
combined_data_log = np.log10(combined_data)
scaler = StandardScaler()
combined_data_scaled = scaler.fit_transform(combined_data_log)

# Split single-spot analyses vs map pixels
spots_data  = combined_data_scaled[combined_source]
pixels_data = combined_data_scaled[~combined_source]
print("Spots :", spots_data.shape)
print("Pixels:", pixels_data.shape)

## 6. Elbow method — choosing the number of clusters K

K-Means is fitted for K = 2 … 10, and the **relative decrease of the within-cluster sum of squares (WSS)** per added cluster is plotted. The "elbow" (where adding clusters stops paying off) supports **K = 6**, the value used in the publication.

In [ ]:
CLUSTER_RANGE = range(2, 11) # candidate numbers of clusters
Ks = list(CLUSTER_RANGE)
kmeans_common = dict(random_state=RANDOM_SEED, n_init=10, max_iter=300)

inertia = []
for K in Ks:
    print(f"Fitting KMeans for K={K}...")
    km = KMeans(n_clusters=K, **kmeans_common).fit(combined_data_scaled)
    inertia.append(km.inertia_)

inertia = np.array(inertia)
rel_drop = -np.diff(inertia) / inertia[:-1]  # relative WSS decrease per added K

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(Ks[1:], rel_drop, marker="o")
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Relative decrease in WSS")
ax.set_title("Elbow method")
ax.grid(linestyle="--", alpha=0.15)
fig.tight_layout()
save_figure(fig, "Elbow_Cluster_LaPalma")
plt.show()

## 7. Final K-Means clustering + PCA visualisation

K-Means (K = 6) is fitted on **all** scaled data. Because K-Means numbers its clusters arbitrarily, the labels are made **stable** by sorting the clusters by their centroid value in the first feature (scaled Cr), so cluster 0 is always the
most Cr-depleted, etc.

A PCA is then fitted **for visualisation only** (the clustering itself uses all eight elements): pixels, single spots, and centroids are shown in PC1–PC2 space.

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_SEED,
                n_init=10, max_iter=300)
combined_clusters = kmeans.fit_predict(combined_data_scaled)
print(f"KMeans complete: {K_FINAL} clusters, "
      f"{combined_data_scaled.shape[0]} points.")

# Stable relabelling: sort clusters by centroid value of feature 0 (scaled Cr)
centroids = kmeans.cluster_centers_
sort_index = np.argsort(centroids[:, 0])
label_map = {old: new for new, old in enumerate(sort_index)}
stable_labels = np.array([label_map[label] for label in combined_clusters])
centroids_stable = centroids[sort_index]

# PCA (visualisation only)
pca = PCA()
pca_scores = pca.fit_transform(combined_data_scaled)
explained_var = pca.explained_variance_ratio_
centroids_pca = pca.transform(centroids_stable)

fig, ax = plot_cluster_scatter(
    pca_scores[:, :2], stable_labels, combined_source, k=K_FINAL,
    centroids_2d=centroids_pca[:, :2],
    xlabel=f"PC1 ({explained_var[0]*100:.1f}% var.)",
    ylabel=f"PC2 ({explained_var[1]*100:.1f}% var.)",
    title=f"PCA space coloured by cluster (K = {K_FINAL})",
    xlim=(-7, 7), ylim=(-7, 7),
)
save_figure(fig, "PCA_Cluster_LaPalma")
plt.show()

## 8. t-SNE visualisation

t-SNE provides a non-linear 2-D embedding as an independent check that the K-means clusters form coherent groups. Because t-SNE is computationally expensive, the map pixels are down-sampled to `N_TSNE_PIXELS` (all single spots are kept).

In [ ]:
N_TSNE_PIXELS = 20000  # map pixels are down-sampled to this many points
rng_tsne = np.random.default_rng(RANDOM_SEED)

labels_spot  = stable_labels[combined_source]
labels_pixel = stable_labels[~combined_source]

if pixels_data.shape[0] > N_TSNE_PIXELS:
    idx = rng_tsne.choice(pixels_data.shape[0], N_TSNE_PIXELS, replace=False)
    X_pixel_tsne, labels_pixel_tsne = pixels_data[idx], labels_pixel[idx]
else:
    X_pixel_tsne, labels_pixel_tsne = pixels_data, labels_pixel

# Sampled pixels first, then all single spots
tsne_data   = np.vstack([X_pixel_tsne, spots_data])
tsne_labels = np.concatenate([labels_pixel_tsne, labels_spot])
tsne_source = np.concatenate([np.zeros(X_pixel_tsne.shape[0], dtype=bool),   # False = map pixel
                              np.ones(spots_data.shape[0], dtype=bool),      # True  = single spot
])

tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto",
            init="pca", max_iter=1000, random_state=RANDOM_SEED)
tsne_results = tsne.fit_transform(tsne_data)

fig, ax = plot_cluster_scatter(
    tsne_results, tsne_labels, tsne_source, k=K_FINAL,
    xlabel="t-SNE 1", ylabel="t-SNE 2",
    title=f"t-SNE coloured by cluster (K = {K_FINAL})",
    pixel_alpha=0.9,
)
save_figure(fig, "TSNE_Cluster_LaPalma")
plt.show()

## 9. Clustering stability analysis

Tests whether the six-cluster solution depends on the strong numerical dominance of map pixels over single-spot analyses. We use the K-Means model fitted on the full dataset (pixels + spots) as a reference. Then,  we fit an independent K-Means model on a balanced dataset combining all single-spot analyses (n = 316) with a random subsample of 1000 pixels. The balanced model's six clusters are matched to the reference clusters by nearest-centroid distance. Boxplots show the chemical distribution for each cluster obtained from the full-dataset clustering; the triangles overlaid on each box represent the median chemical composition of the corresponding matched cluster from the balanced-dataset clustering.

*NOTE*: requires `centroids_stable` and `stable_labels` from the full-dataset clustering cell (Section 7) to already be defined in this kernel session.

In [ ]:
# 1. Independent K-means fit on the BALANCED dataset (spots + downsampled pixels)
rng_balanced = np.random.default_rng(RANDOM_SEED)
pixels_raw_ppm = combined_data[~combined_source]
spots_raw_ppm  = combined_data[combined_source]

n_pix_balanced = 1000
idx_pixels_balanced = rng_balanced.choice(pixels_data.shape[0], size=n_pix_balanced, replace=False)

balanced_data_local = np.vstack([spots_data, pixels_data[idx_pixels_balanced]])
balanced_raw_ppm    = np.vstack([spots_raw_ppm, pixels_raw_ppm[idx_pixels_balanced]])

kmeans_BAL = KMeans(n_clusters=K_FINAL, random_state=RANDOM_SEED, n_init=10, max_iter=300)
combined_clusters_BAL = kmeans_BAL.fit_predict(balanced_data_local)
print(f"KMeans complete (Balanced): {K_FINAL} clusters, {balanced_data_local.shape[0]} points.")

centroids_BAL = kmeans_BAL.cluster_centers_
sort_index_BAL = np.argsort(centroids_BAL[:, 0])
label_map_BAL = {old: new for new, old in enumerate(sort_index_BAL)}
stable_labels_BAL = np.array([label_map_BAL[lbl] for lbl in combined_clusters_BAL])
centroids_stable_BAL = centroids_BAL[sort_index_BAL]

# Match balanced clusters onto the full-dataset cluster numbering
dist_bal = cdist(centroids_stable, centroids_stable_BAL)
full_idx_bal, bal_idx = linear_sum_assignment(dist_bal)
bal_to_full_map = {bal_idx[k]: full_idx_bal[k] for k in range(K_FINAL)}
stable_labels_BAL_matched = np.array([bal_to_full_map[lbl] for lbl in stable_labels_BAL])


# 2. Box plots (full dataset) + balanced cluster-median overlay
cluster_colors = {
    '0': '#E69F00', '1': '#7FC9F5', '2': '#009E73',
    '3': '#F0E442', '4': '#0072B2', '5': '#D55E00',
}
label_map = {
    "Cr": "Cr (ppm)", "Sc": "Sc (ppm)", "Zr": "Zr (ppm)", "Mn": "Mn (ppm)",
    "Ni": "Ni (ppm)", "Sr": "Sr (ppm)", "V": "V (ppm)", "Ti": "Ti (ppm)",
}

df_plot = pd.DataFrame(combined_data, columns=ELEMENTS)
df_plot["Cluster"] = stable_labels.astype(str)

sns.set(style="ticks")
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten()
fontsize = 14

for i, element in enumerate(ELEMENTS):
    sns.boxplot(
        data=df_plot, x="Cluster", y=element,
        order=[str(c) for c in range(K_FINAL)],
        hue="Cluster", hue_order=[str(c) for c in range(K_FINAL)],
        palette=cluster_colors, ax=axes[i],
        showfliers=False, linewidth=1, legend=False, dodge=False,
    )

    bal_median = [
        np.median(balanced_raw_ppm[stable_labels_BAL_matched == c, i]) if np.any(stable_labels_BAL_matched == c)
        else np.nan
        for c in range(K_FINAL)
    ]
    axes[i].scatter(range(K_FINAL), bal_median,
                     marker="^", s=90, facecolor="white", edgecolor="black",
                     linewidth=1.4, zorder=5, label="Median - Balanced dataset")

    axes[i].grid(False)
    axes[i].text(0.02, 0.98, string.ascii_uppercase[i], transform=axes[i].transAxes,
                 fontsize=fontsize, fontweight="bold", va="top", ha="left")
    axes[i].set_xlabel("Cluster number", fontsize=16)
    axes[i].set_ylabel(label_map[element], fontsize=16)
    axes[i].tick_params(axis="both", labelsize=14)
    axes[i].xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[i].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x)}"))

axes[0].legend(loc="upper right", fontsize=14)
plt.tight_layout()
save_figure(fig, "BoxPlot_FullVsBalanced_overlay", formats=("pdf", "png"))
plt.show()

## 10. Cluster-coloured crystal maps

Every crystal map is repainted with its cluster labels: each valid pixel gets the colour of its cluster, masked/background pixels are black. The maps are drawn four per figure and saved to `./Figures/`.

The reconstruction relies on the fact that pixels were stacked in sample order in section 5, so `stable_labels` can be sliced back per sample.

In [ ]:
assert n_dropped_nan == 0, (
    f"{n_dropped_nan} rows were dropped as NaN in Section 5 — per-sample "
    "label slicing below assumes none were dropped. Re-derive per-sample "
    "valid masks before continuing, or this will misalign cluster maps."
)

cmap_clusters, norm_clusters = cluster_map_cmap_norm(K_FINAL)

PLOTS_PER_FIG = 4
num_maps = len(cube_info_list)
num_figs = -(-num_maps // PLOTS_PER_FIG)  # ceiling division
pixel_start = 0  # running index into stable_labels

for fig_idx in range(num_figs):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()

    for subplot_idx in range(PLOTS_PER_FIG):
        overall_idx = fig_idx * PLOTS_PER_FIG + subplot_idx
        if overall_idx >= num_maps:
            axes[subplot_idx].axis("off")
            continue

        info = cube_info_list[overall_idx]
        n_valid = info["clustering_data"].shape[0]
        labels = stable_labels[pixel_start: pixel_start + n_valid]
        pixel_start += n_valid

        # Rebuild the 2-D image: valid pixels get their cluster label,
        # masked pixels get the extra "background" bin (= K_FINAL, black).
        full_labels = np.full(info["n_pixels_total"], K_FINAL)
        full_labels[info["valid_indices"]] = labels
        cluster_map = full_labels.reshape(info["n_rows"], info["n_cols"])
        info["cluster_labels_2d"] = cluster_map

        ax = axes[subplot_idx]
        im = ax.imshow(cluster_map, cmap=cmap_clusters, norm=norm_clusters,
                       interpolation="nearest")
        ax.set_title(info["sample"]["title"], fontsize=9)
        ax.axis("off")
        cbar = fig.colorbar(im, ax=ax, ticks=np.arange(0, K_FINAL), shrink=0.7)
        cbar.set_label("Cluster")

    fig.suptitle(f"Cluster maps — figure {fig_idx + 1}/{num_figs}", fontsize=12)
    fig.tight_layout()
    save_figure(fig, f"ClusterMaps_K{K_FINAL}_{fig_idx + 1}")
    plt.show()
    plt.close(fig)